# n-SBC: Basic Usage

This notebook demonstrates how to train, predict, and evaluate the n-SBC classifier using the Cryotherapy dataset.

In [1]:
import numpy as np
from sklearn.model_selection import LeaveOneOut, train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report

from nsbc import NSBCClassifier

## Load data

In [2]:
data = np.loadtxt("../tests/cryotherapy.csv", delimiter=",")
X, y = data[:, :-1], data[:, -1]

feature_names = ["sex", "age", "time", "n_warts", "type", "area"]
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Classes: {np.unique(y)}, Distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

Samples: 90, Features: 6
Classes: [0. 1.], Distribution: {0.0: 42, 1.0: 48}


## Train / Test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = NSBCClassifier(n_value=3, decimals=2)
clf.fit(X_train, y_train)
print(clf)

NSBCClassifier(n_value=3)


## Predict and evaluate

In [4]:
y_pred = clf.predict(X_test)
print(f"Accuracy: {clf.score(X_test, y_test):.2%}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred))

Accuracy: 83.33%
Balanced accuracy: 82.50%

              precision    recall  f1-score   support

         0.0       0.86      0.75      0.80         8
         1.0       0.82      0.90      0.86        10

    accuracy                           0.83        18
   macro avg       0.84      0.82      0.83        18
weighted avg       0.84      0.83      0.83        18



## Predict probabilities

In [6]:
proba = clf.predict_proba(X_test[:4])
for i in range(4):
    print(f"Sample {i}: pred={y_pred[i]:.0f}, proba={proba[i]}")

Sample 0: pred=1, proba=[0.46948357 0.53051643]
Sample 1: pred=0, proba=[0.56331878 0.43668122]
Sample 2: pred=0, proba=[0.5193133 0.4806867]
Sample 3: pred=0, proba=[0.53181818 0.46818182]


## Leave-One-Out Cross-Validation

LOOCV on the full dataset. The reference for this dataset achieves ~92% balanced accuracy with `n_value=3, decimals=2`.

In [7]:
loo = LeaveOneOut()
y_pred_loo = np.zeros_like(y)

for train_idx, test_idx in loo.split(X):
    clf_loo = NSBCClassifier(n_value=3, decimals=2)
    clf_loo.fit(X[train_idx], y[train_idx])
    y_pred_loo[test_idx] = clf_loo.predict(X[test_idx])

ba = balanced_accuracy_score(y, y_pred_loo)
print(f"LOOCV Balanced Accuracy: {ba:.4%}")

LOOCV Balanced Accuracy: 92.2619%
